# Chapter 1B — Forced exploration of the two posterior modes (Experiment 4d)

This targeted notebook follows up the mode-separation analysis of Experiments 4a and 4b.
For three representative failing learned-boundary fits (one per condition), the
corresponding simulated dataset is reconstructed exactly from its recorded seed and the
general model is refitted with eight chains: four initialised at the generating-direction
mode and four at the reflected alternative, using the chain-mean values observed in the
original failing fit.

Because standard sampling does not move between the modes, initialising chains in both
by construction allows the observed-data log-likelihood to be averaged over draws within
each mode, rather than evaluated only at the chain posterior means. The comparison
remains a likelihood comparison between explored neighbourhoods. It is not integrated
posterior mass or a Bayes factor.

The data-generating functions below reproduce the Chapter 1 pipeline exactly, and each
reconstructed dataset is checked against the recorded number of corruptions before
fitting.

In [1]:
# Setup: constants, exact DGP reconstruction, log-likelihood, Stan model.

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import expit
from cmdstanpy import CmdStanModel

SEED = 2026
KAPPA = 5.0
BETA0_TRUE = -0.8

BETA_GEOM = np.array(
    [1.5, -1.2, 0.8, -0.6, 0.0, 0.0],
    dtype=np.float64,
)

ALPHA_AXIS_GEOM = np.array(
    [0, 0, 0, 0, 0, 1.0],
    dtype=np.float64,
)

ALPHA_OBLIQUE_GEOM = np.array(
    [0, 0, 0, 0, 1.0, 1.0],
    dtype=np.float64,
)
ALPHA_OBLIQUE_GEOM /= np.linalg.norm(ALPHA_OBLIQUE_GEOM)

# Same relative path as chapter1_simulation.ipynb, with a fallback search.
GENERAL_STAN_FILE = Path("corruption_model_general_FINAL.stan")

if not GENERAL_STAN_FILE.exists():
    matches = sorted(Path(".").rglob("corruption_model_general_FINAL.stan"))
    if not matches:
        raise FileNotFoundError(
            "corruption_model_general_FINAL.stan not found. "
            "Run this notebook from the same directory as chapter1_simulation.ipynb."
        )
    GENERAL_STAN_FILE = matches[0]

RESULT_DIR_4D = Path("high_replication/replicate_results")
RESULT_DIR_4D.mkdir(parents=True, exist_ok=True)


def simulate_clean_4d(n, rng, beta0=BETA0_TRUE, beta=BETA_GEOM):
    """Covariates and latent responses, matching the Chapter 1 DGP exactly."""
    beta = np.asarray(beta, dtype=np.float64)
    X = np.asarray(
        rng.normal(size=(n, len(beta))),
        dtype=np.float64,
    )
    p = expit(beta0 + X @ beta)
    y_true = rng.binomial(1, p).astype(np.int8)
    return X, y_true


def make_world_4d(n, alpha, c, q, seed, beta=BETA_GEOM, beta0=BETA0_TRUE):
    """Reproduce one learned-boundary dataset from its recorded seed."""
    alpha = np.asarray(alpha, dtype=np.float64)
    alpha = alpha / np.linalg.norm(alpha)

    rng = np.random.default_rng(seed)
    X, y_true = simulate_clean_4d(n, rng, beta0=beta0, beta=beta)

    region = (X @ alpha) > float(c)
    eligible = region & (y_true == 1)
    flips = eligible & (rng.random(n) < float(q))

    y_obs = y_true.copy()
    y_obs[flips] = 0

    return {"X": X, "y_obs": y_obs, "flips": flips, "alpha_true": alpha}


def seed_for_4d(tag, geometry, n, rep):
    """Seed formulas recorded in Experiments 4a and 4b."""
    if tag == "exp4b":
        return SEED + 80000 + n + rep
    return SEED + 40000 + (10000 if geometry == "oblique" else 0) + n + rep


def total_loglik_4d(beta0, beta, alpha, c, q, X, y, kappa=KAPPA):
    """Observed-data log-likelihood under the one-sided soft-boundary model."""
    alpha = np.asarray(alpha, dtype=float)
    alpha = alpha / np.linalg.norm(alpha)

    p_true = expit(beta0 + X @ np.asarray(beta, dtype=float))
    w = q * expit(kappa * (X @ alpha - c))

    lik = np.where(
        y == 1,
        (1.0 - w) * p_true,
        (1.0 - p_true) + w * p_true,
    )

    return float(np.log(np.clip(lik, 1e-300, None)).sum())


model_general_4d = CmdStanModel(stan_file=str(GENERAL_STAN_FILE))

# One representative failing fit per condition. The mode-level (c, q)
# initial values are the chain posterior means observed in the original
# failing fits (Experiment 4c chain-level table). Corruption counts are
# the recorded values used to verify exact dataset reconstruction.
FORCED_CASES = [
    {"tag": "exp4", "geometry": "axis", "n": 6000, "rep": 2,
     "n_corrupted": 190, "alpha_true": ALPHA_AXIS_GEOM,
     "gen_c": 0.854, "gen_q": 0.466, "ref_c": 0.311, "ref_q": 0.676},
    {"tag": "exp4", "geometry": "oblique", "n": 6000, "rep": 0,
     "n_corrupted": 178, "alpha_true": ALPHA_OBLIQUE_GEOM,
     "gen_c": 0.672, "gen_q": 0.405, "ref_c": 0.339, "ref_q": 0.589},
    {"tag": "exp4b", "geometry": "oblique", "n": 12000, "rep": 9,
     "n_corrupted": 353, "alpha_true": ALPHA_OBLIQUE_GEOM,
     "gen_c": 0.844, "gen_q": 0.424, "ref_c": 0.240, "ref_q": 0.634},
]

print(f"Stan model: {GENERAL_STAN_FILE}")
print(f"Cases to run: {len(FORCED_CASES)}")

Stan model: corruption_model_general_FINAL.stan
Cases to run: 3


## Feasibility check

Run the next cell first with `CASES_TO_RUN = FORCED_CASES[:1]`. It should finish in
roughly 10 to 20 minutes, the reconstruction assertion should pass, and every chain
should report `share_draws_in_init_mode` close to 1. Chains remaining in their
initialised mode is the expected outcome and confirms that each mode is a locally
stable attractor. Once the first case behaves, set `CASES_TO_RUN = FORCED_CASES`
and rerun the cell for all three.

In [4]:
# Refit each failing dataset with four chains initialised in each mode.
# Chains 1-4 start at the generating-direction mode and chains 5-8 at the
# reflected alternative.

CASES_TO_RUN = FORCED_CASES  # feasibility check first; then FORCED_CASES
THIN = 4                          # evaluate the log-likelihood on every 4th draw

forced_rows = []

for case in CASES_TO_RUN:
    seed = seed_for_4d(
        case["tag"], case["geometry"], case["n"], case["rep"],
    )

    world = make_world_4d(
        n=case["n"],
        alpha=case["alpha_true"],
        c=0.8,
        q=0.35,
        seed=seed,
    )

    # Fail immediately if the dataset is not reproduced exactly.
    assert int(world["flips"].sum()) == case["n_corrupted"], (
        f"Reconstruction mismatch for {case['tag']} "
        f"{case['geometry']} n={case['n']} rep={case['rep']}: "
        f"{int(world['flips'].sum())} versus {case['n_corrupted']}"
    )

    X = world["X"]
    y = world["y_obs"].astype(int)
    alpha_true = world["alpha_true"]
    p_dim = X.shape[1]

    init_gen = {
        "beta0": BETA0_TRUE,
        "beta": BETA_GEOM.tolist(),
        "alpha_raw": alpha_true.tolist(),
        "c": case["gen_c"],
        "q": case["gen_q"],
    }

    init_ref = {
        "beta0": BETA0_TRUE,
        "beta": BETA_GEOM.tolist(),
        "alpha_raw": (-alpha_true).tolist(),
        "c": case["ref_c"],
        "q": case["ref_q"],
    }

    inits = [init_gen] * 4 + [init_ref] * 4

    label = (
        f"{case['tag']}|{case['geometry']}|"
        f"n={case['n']}|rep={case['rep']}"
    )

    print(f"Fitting {label} with 8 initialised chains")

    fit = model_general_4d.sample(
        data={
            "n": len(y),
            "p": p_dim,
            "X": X.tolist(),
            "y": y.tolist(),
            "kappa": KAPPA,
            "c_prior_mean": 0.0,
            "c_prior_sd": 1.0,
        },
        chains=8,
        parallel_chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=seed + 1,
        adapt_delta=0.99,
        max_treedepth=12,
        inits=inits,
        show_progress=False,
    )

    # Per-variable extraction avoids loading the generated quantities,
    # which are large for n = 12,000. stan_variable returns draws in
    # chain-major order, so the reshape below splits them by chain.
    n_chains, n_draws = 8, 1000
    beta0_d = fit.stan_variable("beta0").reshape(n_chains, n_draws)
    beta_d = fit.stan_variable("beta").reshape(n_chains, n_draws, p_dim)
    alpha_d = fit.stan_variable("alpha").reshape(n_chains, n_draws, p_dim)
    c_d = fit.stan_variable("c").reshape(n_chains, n_draws)
    q_d = fit.stan_variable("q").reshape(n_chains, n_draws)

    for ch in range(n_chains):
        cos = (
            alpha_d[ch] @ alpha_true
        ) / np.linalg.norm(alpha_d[ch], axis=1)

        init_mode = "generating" if ch < 4 else "reflected"
        init_sign = 1.0 if ch < 4 else -1.0
        stay = float((np.sign(cos) == init_sign).mean())

        idx = np.arange(0, n_draws, THIN)
        logliks = np.array([
            total_loglik_4d(
                beta0_d[ch, i],
                beta_d[ch, i],
                alpha_d[ch, i],
                c_d[ch, i],
                q_d[ch, i],
                X,
                y,
            )
            for i in idx
        ])

        forced_rows.append({
            "fit": label,
            "chain": ch + 1,
            "init_mode": init_mode,
            "share_draws_in_init_mode": round(stay, 3),
            "cos_alpha_mean": round(float(cos.mean()), 3),
            "c_mean": round(float(c_d[ch].mean()), 3),
            "q_mean": round(float(q_d[ch].mean()), 3),
            "avg_total_loglik": round(float(logliks.mean()), 2),
        })

forced_table = pd.DataFrame(forced_rows)
display(forced_table)

/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/3492060788.py:52: RuntimeWarning: divide by zero encountered in matmul
  p = expit(beta0 + X @ beta)
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/3492060788.py:52: RuntimeWarning: overflow encountered in matmul
  p = expit(beta0 + X @ beta)
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/3492060788.py:52: RuntimeWarning: invalid value encountered in matmul
  p = expit(beta0 + X @ beta)
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/3492060788.py:65: RuntimeWarning: divide by zero encountered in matmul
  region = (X @ alpha) > float(c)
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/3492060788.py:65: RuntimeWarning: overflow encountered in matmul
  region = (X @ alpha) > float(c)
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/3492060788.py:65: RuntimeWarning: invalid value encountered in matmul
  region = (X @ alpha) > float(c)
13:30:

Fitting exp4|axis|n=6000|rep=2 with 8 initialised chains


13:31:52 - cmdstanpy - INFO - Chain [2] done processing
13:31:52 - cmdstanpy - INFO - Chain [5] start processing
13:31:55 - cmdstanpy - INFO - Chain [4] done processing
13:31:55 - cmdstanpy - INFO - Chain [6] start processing
13:32:00 - cmdstanpy - INFO - Chain [3] done processing
13:32:00 - cmdstanpy - INFO - Chain [7] start processing
13:32:18 - cmdstanpy - INFO - Chain [1] done processing
13:32:18 - cmdstanpy - INFO - Chain [8] start processing
13:34:28 - cmdstanpy - INFO - Chain [5] done processing
13:34:31 - cmdstanpy - INFO - Chain [6] done processing
13:34:34 - cmdstanpy - INFO - Chain [7] done processing
13:34:46 - cmdstanpy - INFO - Chain [8] done processing
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/461991548.py:93: RuntimeWarning: divide by zero encountered in matmul
  alpha_d[ch] @ alpha_true
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/461991548.py:93: RuntimeWarning: overflow encountered in matmul
  alpha_d[ch] @ alpha_true
/var/f

Fitting exp4|oblique|n=6000|rep=0 with 8 initialised chains


13:37:44 - cmdstanpy - INFO - Chain [3] done processing
13:37:44 - cmdstanpy - INFO - Chain [5] start processing
13:37:44 - cmdstanpy - INFO - Chain [2] done processing
13:37:44 - cmdstanpy - INFO - Chain [6] start processing
13:37:59 - cmdstanpy - INFO - Chain [4] done processing
13:37:59 - cmdstanpy - INFO - Chain [7] start processing
13:38:05 - cmdstanpy - INFO - Chain [1] done processing
13:38:05 - cmdstanpy - INFO - Chain [8] start processing
13:40:41 - cmdstanpy - INFO - Chain [8] done processing
13:40:49 - cmdstanpy - INFO - Chain [6] done processing
13:41:00 - cmdstanpy - INFO - Chain [5] done processing
13:41:09 - cmdstanpy - INFO - Chain [7] done processing
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/461991548.py:93: RuntimeWarning: divide by zero encountered in matmul
  alpha_d[ch] @ alpha_true
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/461991548.py:93: RuntimeWarning: overflow encountered in matmul
  alpha_d[ch] @ alpha_true
/var/f

Fitting exp4b|oblique|n=12000|rep=9 with 8 initialised chains


13:49:05 - cmdstanpy - INFO - Chain [1] done processing
13:49:05 - cmdstanpy - INFO - Chain [5] start processing
13:49:31 - cmdstanpy - INFO - Chain [2] done processing
13:49:31 - cmdstanpy - INFO - Chain [6] start processing
13:49:41 - cmdstanpy - INFO - Chain [3] done processing
13:49:41 - cmdstanpy - INFO - Chain [7] start processing
13:50:26 - cmdstanpy - INFO - Chain [4] done processing
13:50:26 - cmdstanpy - INFO - Chain [8] start processing
13:58:34 - cmdstanpy - INFO - Chain [7] done processing
13:58:43 - cmdstanpy - INFO - Chain [6] done processing
13:59:12 - cmdstanpy - INFO - Chain [8] done processing
13:59:15 - cmdstanpy - INFO - Chain [5] done processing
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/461991548.py:93: RuntimeWarning: divide by zero encountered in matmul
  alpha_d[ch] @ alpha_true
/var/folders/hk/s02zbvd931b5x94gx2t9sh940000gn/T/ipykernel_53283/461991548.py:93: RuntimeWarning: overflow encountered in matmul
  alpha_d[ch] @ alpha_true
/var/f

,fit,chain,init_mode,share_draws_in_init_mode,cos_alpha_mean,c_mean,q_mean,avg_total_loglik
0,exp4|axis|n=6000|rep=2,1,generating,1.0,0.987,0.854,0.466,-2542.05
1,exp4|axis|n=6000|rep=2,2,generating,1.0,0.987,0.857,0.464,-2541.71
2,exp4|axis|n=6000|rep=2,3,generating,1.0,0.987,0.870,0.463,-2542.21
3,exp4|axis|n=6000|rep=2,4,generating,1.0,0.987,0.860,0.462,-2542.06
4,exp4|axis|n=6000|rep=2,5,reflected,1.0,-0.335,0.318,0.672,-2587.40
5,exp4|axis|n=6000|rep=2,6,reflected,1.0,-0.327,0.306,0.679,-2587.25
6,exp4|axis|n=6000|rep=2,7,reflected,1.0,-0.328,0.303,0.678,-2587.33
7,exp4|axis|n=6000|rep=2,8,reflected,1.0,-0.328,0.303,0.679,-2587.11
8,exp4|oblique|n=6000|rep=0,1,generating,1.0,0.986,0.676,0.406,-2617.69
9,exp4|oblique|n=6000|rep=0,2,generating,1.0,0.986,0.673,0.408,-2617.78


In [3]:
# Mode-level summary and the average-over-draws log-likelihood gap.

mode_summary = (
    forced_table
    .groupby(["fit", "init_mode"], as_index=False)
    .agg(
        cos_alpha_mean=("cos_alpha_mean", "mean"),
        c_mean=("c_mean", "mean"),
        q_mean=("q_mean", "mean"),
        avg_total_loglik=("avg_total_loglik", "mean"),
        min_share_in_init_mode=("share_draws_in_init_mode", "min"),
    )
)

display(mode_summary)

gaps = (
    mode_summary
    .pivot(index="fit", columns="init_mode", values="avg_total_loglik")
    .assign(
        gap_generating_minus_reflected=lambda d: (
            d["generating"] - d["reflected"]
        )
    )
)

print("Average-over-draws log-likelihood gap (generating minus reflected):")
print(gaps.round(2).to_string())

forced_table.to_csv(
    RESULT_DIR_4D / "exp4d_forced_exploration_per_chain.csv",
    index=False,
)
mode_summary.to_csv(
    RESULT_DIR_4D / "exp4d_forced_exploration_by_mode.csv",
    index=False,
)

print("\nResults written to", RESULT_DIR_4D.resolve())

,fit,init_mode,cos_alpha_mean,c_mean,q_mean,avg_total_loglik,min_share_in_init_mode
0,exp4|axis|n=6000|rep=2,generating,0.9870,0.86025,0.46375,-2542.0075,1.0
1,exp4|axis|n=6000|rep=2,reflected,-0.3295,0.30750,0.67700,-2587.2725,1.0


Average-over-draws log-likelihood gap (generating minus reflected):
init_mode               generating  reflected  gap_generating_minus_reflected
fit                                                                          
exp4|axis|n=6000|rep=2    -2542.01   -2587.27                           45.26

Results written to /Users/demikramer/Desktop/MLDS_Demi_Kramer_Imperial copy/high_replication/replicate_results


## Interpretation notes

The quantity reported per mode is the observed-data log-likelihood averaged over that
mode's retained draws, so the comparison no longer rests only on the chain posterior
means. It remains a comparison of likelihood between the two explored neighbourhoods
and does not measure integrated posterior mass, the volume surrounding each mode, or a
Bayes factor. `share_draws_in_init_mode` records whether any chain migrated between
modes during sampling. Values near 1 for every chain confirm that each mode is a
locally stable attractor for the sampler, consistent with the cross-chain separation
observed in Experiments 4a and 4b.